# EgoDex × Daft — hand-pose pipeline, step by step

**Build** the feature + embedding datasets, then **query** them. Run top to bottom.

**Kernel:** `Python (egodex .venv)` — has the `Hdf5File` Daft build + torch/av/h5py.

## 0. Setup

In [ ]:
!uv pip install -q --upgrade "daft[transformers, hdf5, video]" --pre --extra-index-url https://nightly.daft.ai

In [ ]:
import os

RAW_HDF5 = ".data/*/*.hdf5"  # populated by the download cell below
DATASET = "egodex_lerobot_full"  # the LeRobot dataset already on disk
EPISODES = [0, 1, 4, 6]  # a small subset so each cell runs in seconds

os.environ["DAFT_PROGRESS_BAR"] = "0"  # quiet Daft progress bars
os.environ["TQDM_DISABLE"] = "1"  # quiet model loading bars

## 1. Convert raw EgoDex HDF5 → LeRobot  *(optional)*

Daft reads HDF5 natively via the `Hdf5File` type. Needs raw `.hdf5` + `pip install lerobot` for the write. **Already have the dataset? Skip this cell.**

First star by downloading the test dataset. This will take ~20 minutes depending on your internet speed.

In [ ]:
import subprocess
from pathlib import Path

DATA_DIR = Path(".data")
ZIP_PATH = DATA_DIR / "test.zip"
URL = "https://ml-site.cdn-apple.com/datasets/egodex/test.zip"


def has_egodex_data() -> bool:
    """Fast check: any task folder under .data/ already contains HDF5 episodes."""
    if not DATA_DIR.is_dir():
        return False
    return any(child.is_dir() and any(child.glob("*.hdf5")) for child in DATA_DIR.iterdir())


if has_egodex_data():
    print(f"EgoDex data already present under {DATA_DIR.resolve()} — skipping download.")
elif ZIP_PATH.exists():
    print(f"Found cached zip at {ZIP_PATH} — extracting into {DATA_DIR.resolve()} ...")
    subprocess.run(["unzip", "-o", str(ZIP_PATH), "-d", str(DATA_DIR)], check=True)
    print("Done.")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Downloading test.zip (~16 GB) to {ZIP_PATH} ...")
    subprocess.run(["curl", "-L", URL, "-o", str(ZIP_PATH)], check=True)
    print(f"Extracting into {DATA_DIR.resolve()} ...")
    subprocess.run(["unzip", "-o", str(ZIP_PATH), "-d", str(DATA_DIR)], check=True)
    print("Done.")

100 16.1G  100 16.1G    0     0  16.9M      0  0:16:11  0:16:11 --:--:-- 18.8M      0  0:15:58  0:08:59  0:06:59 15.1M      0  0:15:59  0:09:06  0:06:53 15.9M17.1M      0  0:15:59  0:09:24  0:06:35 19.1M 0  17.2M      0  0:15:59  0:09:27  0:06:32 19.4M0:09:30  0:06:28 19.2M   0  0:15:58  0:09:34  0:06:24 17.2M   0  0:15:59  0:09:42  0:06:17 17.4M.0M      0  0:16:05  0:10:34  0:05:31 17.8M7.0M      0  0:16:05  0:10:37  0:05:28 18.3M 0  0:16:05  0:10:47  0:05:18 17.1M7.0M      0  0:16:05  0:10:52  0:05:13 16.7M 0  17.0M      0  0:16:06  0:10:55  0:05:11 14.2M0  16.9M      0  0:16:13  0:11:17  0:04:56 18.2M0     0  16.9M      0  0:16:15  0:11:22  0:04:53 12.7M16.9M      0  0:16:15  0:11:25  0:04:50 16.2M0  16.9M      0  0:16:15  0:11:49  0:04:26 17.3M16.9M      0  0:16:14  0:13:53  0:02:21 18.1M9M      0  0:16:14  0:14:00  0:02:14 16.3M  0     0  16.9M      0  0:16:15  0:14:07  0:02:08 17.1M  16.9M      0  0:16:15  0:14:22  0:01:53 18.9M0 18.3M 0:14:27  0:01:47 17.8M  0  16.9M      0  0:1

In [ ]:
from edodex_lib import convert_egodex_to_lerobot

# Convert raw EgoDex HDF5 → LeRobot
lerobot_dir = convert_egodex_to_lerobot(
    hdf5_glob=os.path.expanduser(RAW_HDF5), 
    repo_id="egodex", 
    output_dir="egodex_converted/"
)

# Build
Read the dataset, then compute the pose features (30 fps) **and** the SigLIP embeddings (~1 fps) — both before any querying.

## 2. Read the LeRobot dataset

One row per frame. Schema first, then a few rows (with frame thumbnails).

In [ ]:
df = daft.datasets.lerobot.read(DATASET, load_video_frames="observation.image").where(
    col("episode_index").is_in(EPISODES)
)
display(df.schema())
df.show(3)

## 3. Embed frames:  `embed_frames`  *(the semantic embedding step)*

Here you will run SigLIP-2 as a batched `@daft.cls` UDF. This writes a unit-normalized `clip_emb` column (~1 fps). 

In [ ]:
from daft import DataType, Series
from transformers import AutoModel, AutoProcessor
import torch


def _auto_device() -> str:
    if _HAS_CUDA:
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


_HAS_CUDA = torch.cuda.is_available()
GPUS = 1 if _HAS_CUDA else 0

DEVICE = os.environ.get("CLIP_DEVICE", _auto_device())
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

SUBSAMPLE = 30  # keep 1 of every 30 frames (~1 fps); semantic content barely changes between adjacent frames
MODEL_ID = "google/siglip2-base-patch16-224"
EMB_DIM = 768  # SigLIP2-base shared image/text embedding dim (must match the model)

In [ ]:
# --- SigLIP embedding ------------------------------------------------------
def _normalized_embedding(model_output) -> torch.Tensor:
    """Pull the embedding tensor out of a transformers output and L2-normalize it.

    transformers 5.x returns a model-output object from get_image_features /
    get_text_features; older versions returned a bare tensor. Handle both.
    """
    if torch.is_tensor(model_output):
        feats = model_output
    else:
        feats = model_output.pooler_output
    feats = feats.float()
    return feats / feats.norm(dim=-1, keepdim=True)  # unit-norm so cosine == dot product


@daft.cls(gpus=GPUS, max_concurrency=1, use_process=False)
class SiglipEmbedder:
    """Encode each frame into a unit-norm SigLIP image embedding.

    Single-node, in-process: runs on this machine's device (CUDA on the EC2 GPU
    box, CPU/MPS for a local smoke test). The model loads once per instance.
    """

    def __init__(self) -> None:
        self.model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE).eval()
        self.processor = AutoProcessor.from_pretrained(MODEL_ID)

    @daft.method.batch(return_dtype=DataType.embedding(DataType.float32(), EMB_DIM), batch_size=16)
    def embed_image(self, images: Series):
        # images.to_pylist() yields uint8 H×W×C numpy arrays; the SigLIP processor takes them
        # directly (verified identical to the PIL path), so no per-frame Image.fromarray needed.
        inputs = self.processor(images=images.to_pylist(), return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            model_output = self.model.get_image_features(**inputs)
            embeddings = _normalized_embedding(model_output)
        return list(embeddings.cpu().numpy())

In [ ]:
emb = (
    egodex.embed_frames(
        daft.datasets.lerobot.read(DATASET, load_video_frames="observation.image").where(
            col("episode_index").is_in(EPISODES)
        )
    )
    .select("episode_index", "frame_index", "clip_emb")
    .collect()
)
emb.show(3)  # one 768-d embedding per ~1 fps frame

## 4. Per-frame geometry: `add_state_features`

A row-wise Daft UDF emits continuous geometry: closure, finger flexion, thumb distances, etc. This is a core feature of Daft's lazy execution model. Run the cell below to see the computed static features!

In [ ]:
df = egodex.add_state_features(df)
df.show(3)

## 5. Action rates over frames  →  `add_skeleton_features`

Daft window functions and Daft's `euclidean_distance` differentiate static features into motion: `curl_rate`, `wrist_speed`, `roll`, etc. Run the cell below to see the computed motion features!

In [ ]:
df = egodex.add_skeleton_features(df)
df.show(3)

## 6. Write the feature dataset to Parquet for future use!

In [ ]:
df.write_parquet("features_demo/")
features = daft.read_parquet("features_demo/")
print("rows:", features.count_rows())
features.show(3)

# Query
The features + embeddings are precomputed and saved. Now just query them.

## 7. Calibrate the scenario thresholds (once)

Data-driven cut points via Daft `.percentile`. These are computed once and reused by every query. Run the cell below to see the thresholds!

In [ ]:
thresholds = egodex.calibrate(features)
thresholds

## 8. Query by a hand-pose scenario

Ranks episodes by matching-frame count and returns the top hits. Run the cell below to see the hits that we matched to!

In [ ]:
hits = egodex.query(features, pose="openness", open_lo=0.9, open_hi=1.0, k=5, thresholds=thresholds)
hits

## 9. Query by semantic text

Pass in the matching string that you want to see in the frames. For now we query for shirts. 

In [ ]:
egodex.query(emb, text="shirts", k=2)

## 10. Visualize a match — skeleton overlay

In [ ]:
top = hits[0]
frame = top["segments"][0][0]
egodex.overlay(DATASET, top["episode_index"], frame)

## 11. Explore interactively (Gradio)

Launch the full **query UI** — the same `query_ui.py` dashboard from the Results section (looping match clips, segment stepper, live skeleton overlay, per-frame pose table). It returns a public `*.gradio.live` link. Set `DATASET` first: `query_ui` reads it at import to warm up the embeddings, episode metadata, and raw pose for the overlay.

In [ ]:
# The interactive UI *is* query_ui.py — import it and launch that exact app for a live link.
# query_ui reads DATASET at import time to warm up (embeddings, episode metadata, overlay pose),
# so set it first. Returns a public *.gradio.live URL.
import os

os.environ["DATASET"] = DATASET
import query_ui

query_ui.demo.launch(share=True, allowed_paths=[query_ui.CLIPS_DIR, query_ui.FRAMES_DIR])